In [ ]:
import psycopg2
import os
import pandas as pd
import yaml
import re

In [ ]:
os.environ["PGSERVICEFILE"] = os.path.join(os.getcwd(), ".pg_service.conf")
os.environ["PGPASSFILE"] = os.path.join(os.getcwd(), ".pgpass")

with open(os.environ["PGPASSFILE"]) as f:
    pgpass = f.read().strip()

In [ ]:
beaver = psycopg2.connect(service="beam_neb0",password=pgpass)
cursor = beaver.cursor()

In [ ]:
## schema
cursor.execute("""
    select
        c.table_schema,
        c.table_name,
        c.column_name,
        c.data_type,
        c.is_nullable,
        c.column_default,
        pgd.description AS column_comment
    from information_schema.columns c
    left join pg_catalog.pg_statio_all_tables st
        on st.schemaname = c.table_schema
        and st.relname = c.table_name
    left join pg_catalog.pg_description pgd
        ON pgd.objoid = st.relid
        AND pgd.objsubid = c.ordinal_position
    where c.table_schema = 'public'
    order by c.table_schema, c.table_name, c.ordinal_position;
""")

columns_df = pd.DataFrame(cursor.fetchall(), columns=[
    "schema", "table", "column", "data_type", "nullable", "default", "comment"])

In [ ]:
output_dir = os.path.join(os.getcwd(), "SL")
os.makedirs(output_dir, exist_ok=True)

def detect_pk(group):
    """Detect primary key from serial/nextval default."""
    for _, row in group.iterrows():
        default = str(row["default"]) if pd.notna(row["default"]) else ""
        if "nextval(" in default:
            return row["column"]
    return None

def detect_fks(group):
    """Detect foreign keys from comment text containing 'Foreign key'."""
    fks = []
    for _, row in group.iterrows():
        comment = str(row["comment"]) if pd.notna(row["comment"]) else ""
        if "foreign key" in comment.lower():
            # Extract referenced table from comment like "Foreign key linking to the users table"
            match = re.search(r'(?:linking to|referencing|references)\s+(?:the\s+)?(\w+)', comment, re.IGNORECASE)
            if match:
                ref_table = match.group(1)
                fks.append({
                    "field": row["column"],
                    "ref_table": ref_table,
                    "ref_field": row["column"],  # FK column name matches referenced PK
                })
    return fks

def generate_measures(group, pk_col):
    """Generate measures based on column types."""
    measures = []
    fk_cols = {row["column"] for _, row in group.iterrows()
               if pd.notna(row["comment"]) and "foreign key" in str(row["comment"]).lower()}

    for _, row in group.iterrows():
        col = row["column"]
        dtype = row["data_type"]

        # PK → count measure
        if col == pk_col:
            table = row["table"]
            measures.append({
                "name": f"{col}_count",
                "formula": f"count({col})",
                "description": f"Total number of records in {table}.",
            })
        # Numeric/non-FK integer → sum and avg
        elif dtype == "numeric" or (dtype == "integer" and col not in fk_cols and col != pk_col):
            measures.append({
                "name": f"total_{col}",
                "formula": f"sum({col})",
                "description": f"Sum of {col} across all records.",
            })
            measures.append({
                "name": f"avg_{col}",
                "formula": f"avg({col})",
                "description": f"Average {col} across all records.",
            })
        # Timestamp → min and max
        elif "timestamp" in dtype:
            measures.append({
                "name": f"earliest_{col}",
                "formula": f"min({col})",
                "description": f"Earliest {col}.",
            })
            measures.append({
                "name": f"latest_{col}",
                "formula": f"max({col})",
                "description": f"Most recent {col}.",
            })
    return measures

def build_yaml_string(table_name, description, pk, fks, fields, measures):
    """Build YAML string matching the reference format."""
    lines = []
    lines.append(f"table: {table_name}")
    lines.append(f"description: {description}")
    lines.append("")
    lines.append(f"primary_key: {pk}" if pk else "primary_key:")
    lines.append("")

    # Relationships
    if fks:
        lines.append("relationships:")
        for fk in fks:
            lines.append(f"  - field: {fk['field']}")
            lines.append(f"    references:")
            lines.append(f"      table: {fk['ref_table']}")
            lines.append(f"      field: {fk['ref_field']}")
            lines.append(f"    type: many_to_one")
    else:
        lines.append("relationships: []")
    lines.append("")

    # Fields
    lines.append("fields:")
    for f in fields:
        lines.append(f"  - name: {f['name']}")
        lines.append(f"    description: {f['description']}")
        lines.append(f"    data_type: {f['data_type']}")
        lines.append(f"    nullable: {str(f['nullable']).lower()}")
        lines.append("")

    # Measures
    if measures:
        lines.append("measures:")
        for m in measures:
            lines.append(f"  - name: {m['name']}")
            lines.append(f"    formula: {m['formula']}")
            lines.append(f"    description: {m['description']}")
            lines.append("")

    return "\n".join(lines) + "\n"

# Generate a YML file per table
for table_name, group in columns_df.groupby("table"):
    pk = detect_pk(group)
    fks = detect_fks(group)

    fields = []
    for _, row in group.iterrows():
        comment = str(row["comment"]) if pd.notna(row["comment"]) else ""
        fields.append({
            "name": row["column"],
            "description": comment.lower().rstrip(".") + "." if comment else "",
            "data_type": row["data_type"],
            "nullable": row["nullable"] == "YES",
        })

    measures = generate_measures(group, pk)

    description = f"Each row represents a record in the {table_name} table."

    yml_content = build_yaml_string(table_name, description, pk, fks, fields, measures)

    filepath = os.path.join(output_dir, f"{table_name}.yml")
    with open(filepath, "w") as f:
        f.write(yml_content)

print(f"Generated {len(columns_df['table'].unique())} YML files in {output_dir}/")